<p align="center"><img src="https://raw.githubusercontent.com/arab-future-academy/deep_dive_in_gen_ai/refs/heads/main/imgs/arabfutureacademy.png" alt="Arabic Future Academy" width="720" /></p>

# كومفي يو أي 

In [ ]:
#@title 0. Central Configuration
#@markdown Set the main options here before running the rest of the notebook.

# Storage and updates
MOUNT_DRIVE = True #@param {type:"boolean"}
FORCE_REMOUNT_DRIVE = False #@param {type:"boolean"}
UPDATE_COMFY_UI = True #@param {type:"boolean"}
INSTALL_COMFYUI_MANAGER = True #@param {type:"boolean"}

# Paths
LOCAL_WORKSPACE = "/content/ComfyUI" #@param {type:"string"}
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI" #@param {type:"string"}

# Runtime checks
RUN_HEALTH_CHECKS = True #@param {type:"boolean"}
MIN_FREE_DISK_GB = 15 #@param {type:"integer"}

CACHE_TAR = f"{DRIVE_WORKSPACE}/comfy_ui_cache.tar"

print("Configuration loaded.")
print(f"Local workspace: {LOCAL_WORKSPACE}")
print(f"Drive workspace: {DRIVE_WORKSPACE}")
print(f"Cache archive: {CACHE_TAR}")

In [ ]:
#@title 1. System Initialization
#@markdown **Run this cell first.** <br>
#@markdown Mounts Drive, sets up ComfyUI, and extracts your high-speed Tarball cache to the local SSD for instant booting.

import os
import shutil
import subprocess
import sys
from google.colab import drive

# Defaults allow this cell to run even if the config cell was skipped.
MOUNT_DRIVE = globals().get("MOUNT_DRIVE", True)
FORCE_REMOUNT_DRIVE = globals().get("FORCE_REMOUNT_DRIVE", False)
UPDATE_COMFY_UI = globals().get("UPDATE_COMFY_UI", True)
INSTALL_COMFYUI_MANAGER = globals().get("INSTALL_COMFYUI_MANAGER", True)
LOCAL_WORKSPACE = globals().get("LOCAL_WORKSPACE", "/content/ComfyUI")
DRIVE_WORKSPACE = globals().get("DRIVE_WORKSPACE", "/content/drive/MyDrive/ComfyUI")
RUN_HEALTH_CHECKS = globals().get("RUN_HEALTH_CHECKS", True)
MIN_FREE_DISK_GB = globals().get("MIN_FREE_DISK_GB", 15)
CACHE_TAR = globals().get("CACHE_TAR", os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar"))

print("[SYSTEM] Initialization started.\n")

def stream_cmd(cmd, cwd=None):
    """Streams shell commands directly to the output"""
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd, bufsize=1)
    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()

def run_health_checks(stage):
    if not RUN_HEALTH_CHECKS:
        return

    print(f"\n[HEALTH] {stage}")

    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    if gpu.returncode == 0 and gpu.stdout.strip():
        print(f"[OK] GPU: {gpu.stdout.strip()}")
    else:
        print("[WARN] No NVIDIA GPU detected. In Colab, set Runtime > Change runtime type > GPU.")

    total, used, free = shutil.disk_usage("/content")
    free_gb = free / (1024 ** 3)
    print(f"[OK] /content free disk: {free_gb:.1f} GB")
    if free_gb < MIN_FREE_DISK_GB:
        print(f"[WARN] Free disk is below {MIN_FREE_DISK_GB} GB. Large model downloads may fail.")

    if MOUNT_DRIVE:
        drive_root = "/content/drive/MyDrive"
        if os.path.exists(drive_root):
            print(f"[OK] Google Drive mounted: {drive_root}")
        else:
            print("[INFO] Google Drive is not mounted yet.")

    if os.path.exists(LOCAL_WORKSPACE):
        print(f"[OK] Local workspace exists: {LOCAL_WORKSPACE}")
    else:
        print(f"[INFO] Local workspace will be created: {LOCAL_WORKSPACE}")

def mount_google_drive():
    drive_root = "/content/drive/MyDrive"
    if os.path.exists(drive_root) and not FORCE_REMOUNT_DRIVE:
        print(f"[OK] Google Drive already mounted: {drive_root}")
        return

    try:
        drive.mount('/content/drive', force_remount=FORCE_REMOUNT_DRIVE)
    except Exception as err:
        if not FORCE_REMOUNT_DRIVE:
            print("[DRIVE] First mount failed. Retrying once with force_remount=True...")
            try:
                drive.mount('/content/drive', force_remount=True)
            except Exception as retry_err:
                raise RuntimeError(
                    "Google Drive mount failed. In Colab, try Runtime > Restart runtime, "
                    "then run the configuration and initialization cells again. Also check that "
                    "browser popups/cookies are allowed for colab.research.google.com. "
                    f"Original error: {err}; retry error: {retry_err}"
                )
        else:
            raise RuntimeError(
                "Google Drive mount failed even with force_remount enabled. Try Runtime > Restart runtime, "
                "then run the cells again and complete the Google authorization popup. "
                f"Original error: {err}"
            )

    if not os.path.exists(drive_root):
        raise RuntimeError("Google Drive mount finished, but /content/drive/MyDrive was not found.")

run_health_checks("Before setup")

# 1. Mount Google Drive
if MOUNT_DRIVE:
    print("[DRIVE] Mounting Google Drive...")
    mount_google_drive()
    os.makedirs(DRIVE_WORKSPACE, exist_ok=True)
    run_health_checks("After Drive mount")

# 2. Setup ComfyUI Core
if not os.path.exists(LOCAL_WORKSPACE):
    print("[COMFYUI] Cloning ComfyUI...")
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", LOCAL_WORKSPACE])
else:
    if UPDATE_COMFY_UI:
        print("[COMFYUI] Checking for updates...")
        subprocess.run(["git", "pull"], cwd=LOCAL_WORKSPACE)

# 3. Configure Hybrid Storage & Tarball Cache
heavy_dirs = ["models", "output", "input"]
print("\n[STORAGE] Routing heavy model directories directly to Drive...")
for d in heavy_dirs:
    local_path = os.path.join(LOCAL_WORKSPACE, d)
    drive_path = os.path.join(DRIVE_WORKSPACE, d)
    if not os.path.exists(drive_path): os.makedirs(drive_path, exist_ok=True)
    if os.path.exists(local_path) and not os.path.islink(local_path): shutil.rmtree(local_path)
    if not os.path.exists(local_path): os.symlink(drive_path, local_path)

print("\n[CACHE] Deploying high-speed tarball cache for UI elements...")
if os.path.exists(CACHE_TAR):
    print("   Extracting cached UI nodes to local SSD...")
    os.system(f"tar -xf '{CACHE_TAR}' -C '{LOCAL_WORKSPACE}' > /dev/null 2>&1")
else:
    print("   No cache found on Drive. Creating fresh local directories...")
    os.makedirs(os.path.join(LOCAL_WORKSPACE, "custom_nodes"), exist_ok=True)
    os.makedirs(os.path.join(LOCAL_WORKSPACE, "user"), exist_ok=True)

# 4. Install ComfyUI Manager
if INSTALL_COMFYUI_MANAGER:
    manager_path = os.path.join(LOCAL_WORKSPACE, "custom_nodes", "ComfyUI-Manager")
    if not os.path.exists(manager_path):
        print("\n[MANAGER] Installing ComfyUI Manager...")
        subprocess.run(["git", "clone", "https://github.com/ltdrdata/ComfyUI-Manager.git", manager_path])
    else:
        print("\n[OK] ComfyUI Manager verified.")
        subprocess.run(["git", "pull"], cwd=manager_path)

# 5. Core Python Dependencies
print("\n[PYTHON] Installing base Python requirements...")
stream_cmd(["pip", "install", "xformers!=0.0.18", "-r", "requirements.txt", "--extra-index-url", "https://download.pytorch.org/whl/cu121"], cwd=LOCAL_WORKSPACE)
stream_cmd(["pip", "install", "insightface", "onnxruntime-gpu"])

# 6. AUTO-HEAL: Custom Node Dependencies
print("\n[AUTO-HEAL] Scanning local custom nodes for missing dependencies...")
custom_nodes_dir = os.path.join(LOCAL_WORKSPACE, "custom_nodes")
if os.path.exists(custom_nodes_dir):
    for item in os.listdir(custom_nodes_dir):
        node_path = os.path.join(custom_nodes_dir, item)
        req_file = os.path.join(node_path, "requirements.txt")
        if os.path.isdir(node_path) and os.path.exists(req_file):
            print(f"\n   Restoring dependencies for: {item}")
            stream_cmd(["pip", "install", "-r", "requirements.txt"], cwd=node_path)

run_health_checks("After setup")
print("\n[SYSTEM READY] Initialization complete.")

In [ ]:
#@title 2. Model & Node Downloader
#@markdown Paste download links below. This supports **Checkpoints, Diffusion Models (Flux/UNET), Text Encoders (CLIP/T5), CLIP Vision, VAEs, LoRAs, and ControlNets**.

import os
import requests
from urllib.parse import urlparse, unquote
from tqdm.auto import tqdm

WORKSPACE = "/content/ComfyUI"

# --- Input Resources ---
CHECKPOINT_URLS = "" #@param {type:"string"}
UNET_DIFFUSION_URLS = "" #@param {type:"string"}
TEXT_ENCODER_URLS = "" #@param {type:"string"}
CLIP_VISION_URLS = "" #@param {type:"string"}
VAE_URLS = "" #@param {type:"string"}
LORA_URLS = "" #@param {type:"string"}
CONTROLNET_URLS = "" #@param {type:"string"}
UPSCALE_MODELS_URLS = "" #@param {type:"string"}
EMBEDDING_URLS = "" #@param {type:"string"}
CUSTOM_NODE_URLS = "" #@param {type:"string"}

# --- Downloader Logic ---
DIRS = {
    "checkpoints":    os.path.join(WORKSPACE, "models/checkpoints"),
    "unet":           os.path.join(WORKSPACE, "models/unet"),
    "clip":           os.path.join(WORKSPACE, "models/clip"),
    "clip_vision":    os.path.join(WORKSPACE, "models/clip_vision"),
    "vae":            os.path.join(WORKSPACE, "models/vae"),
    "loras":          os.path.join(WORKSPACE, "models/loras"),
    "controlnet":     os.path.join(WORKSPACE, "models/controlnet"),
    "upscale_models": os.path.join(WORKSPACE, "models/upscale_models"),
    "embeddings":     os.path.join(WORKSPACE, "models/embeddings"),
    "custom_nodes":   os.path.join(WORKSPACE, "custom_nodes")
}

def get_filename(url, response):
    """Smartly determines filename from Content-Disposition or URL."""
    if "Content-Disposition" in response.headers:
        import re
        fname = re.findall('filename="?([^"]+)"?', response.headers["Content-Disposition"])
        if fname: return fname[0]
    return unquote(os.path.basename(urlparse(url).path))

def download_file(url, target_dir):
    try:
        # Stream the download to get headers first
        response = requests.get(url, stream=True, allow_redirects=True)
        response.raise_for_status()

        filename = get_filename(url, response)
        file_path = os.path.join(target_dir, filename)
        total_size = int(response.headers.get('content-length', 0))

        if os.path.exists(file_path):
            print(f"   [SKIP] Exists: {filename}")
            return

        # Modern Progress Bar Log
        print(f"   [DOWNLOAD] {filename}")

        # The Progress Bar (Auto-Stretching)
        with tqdm(
            total=total_size,
            unit='B',
            unit_scale=True,
            unit_divisor=1024,
            desc="      Progress",
            dynamic_ncols=True
        ) as bar:
            with open(file_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024*1024): # 1MB chunks
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))

        print("      [OK] Download complete\n")

    except Exception as e:
        print(f"   [ERROR] Failed to download: {url}")
        print(f"      Error: {e}\n")

def process_downloads(urls_str, target_dir, is_node=False):
    if not urls_str.strip(): return

    url_list = [u.strip() for u in urls_str.replace(',', '\n').split('\n') if u.strip()]
    if not os.path.exists(target_dir): os.makedirs(target_dir, exist_ok=True)

    print(f"[CATEGORY] {os.path.basename(target_dir)}")

    for url in url_list:
        if is_node:
            node_name = url.split('/')[-1].replace('.git', '')
            node_path = os.path.join(target_dir, node_name)
            if not os.path.exists(node_path):
                print(f"   [CLONE] Node: {node_name}...")
                !git clone {url} {node_path}
                # Auto-install requirements
                req = os.path.join(node_path, "requirements.txt")
                if os.path.exists(req):
                    print(f"      Installing requirements...")
                    !pip install -r "{req}"
                print("      [OK] Installed\n")
            else:
                print(f"   [SKIP] Node exists: {node_name}\n")
        else:
            download_file(url, target_dir)

# --- Execution ---
process_downloads(CHECKPOINT_URLS,     DIRS["checkpoints"])
process_downloads(UNET_DIFFUSION_URLS, DIRS["unet"])
process_downloads(TEXT_ENCODER_URLS,   DIRS["clip"])
process_downloads(CLIP_VISION_URLS,    DIRS["clip_vision"])
process_downloads(VAE_URLS,            DIRS["vae"])
process_downloads(LORA_URLS,           DIRS["loras"])
process_downloads(CONTROLNET_URLS,     DIRS["controlnet"])
process_downloads(UPSCALE_MODELS_URLS, DIRS["upscale_models"])
process_downloads(EMBEDDING_URLS,      DIRS["embeddings"])
process_downloads(CUSTOM_NODE_URLS,    DIRS["custom_nodes"], is_node=True)

print("All tasks finished.")

In [ ]:
# @title 3. Session Anti-Disconnect
%%html
<b>Keep-Alive Audio</b><br>
<i>Running this silent audio loop prevents the browser tab from sleeping.</i><br>
<audio src="https://raw.githubusercontent.com/anars/blank-audio/master/10-minutes-of-silence.mp3" autoplay loop controls style="width: 300px;" />

In [ ]:
#@title 4. Start ComfyUI Session
#@markdown Installs missing runtime dependencies, establishes an optimized IPv4 Cloudflare tunnel, and launches the ComfyUI web interface with live standard logs.

import subprocess
import threading
import time
import socket
import os
import sys

# --- Session Configuration ---
#@markdown **Performance Profile:**
MEMORY_PROFILE = "Standard (Auto-Detect)" #@param ["Standard (Auto-Detect)", "Low VRAM (T4 GPU / Heavy Models)", "High VRAM (A100 GPU Only)"]
#@markdown **Visual Settings:**
LIVE_GENERATION_PREVIEWS = True #@param {type:"boolean"}

WORKSPACE = "/content/ComfyUI"
ARGS = []

if "Low VRAM" in MEMORY_PROFILE:
    ARGS.append("--lowvram")
elif "High VRAM" in MEMORY_PROFILE:
    ARGS.append("--highvram")

if LIVE_GENERATION_PREVIEWS:
    ARGS.extend(["--preview-method", "auto"])

# 0. Auto-Heal Missing Custom Node Dependencies
print("[SYSTEM] Auto-healing runtime dependencies...")
os.system("pip install -q gguf piexif > /dev/null 2>&1")

# 1. Initialize Network Tunnel 
print("[SYSTEM] Verifying Cloudflare Daemon...")
if not os.path.exists("/usr/bin/cloudflared"):
    print("[SYSTEM] Fetching Cloudflared via aria2c...")
    os.system("apt-get install -y aria2 > /dev/null 2>&1")
    os.system("aria2c -x 16 -s 16 -k 1M https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -d /content -o cloudflared.deb > /dev/null 2>&1")
    os.system("dpkg -i /content/cloudflared.deb > /dev/null 2>&1")

def start_tunnel(port):
    while True:
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break

    print("\n[SYSTEM] ComfyUI Local Server Detected. Establishing Secure Tunnel...")
    
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}", "--edge-ip-version", "4", "--protocol", "http2"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    tunnel_established = False
    for line in p.stderr:
        if "trycloudflare.com" in line and not tunnel_established:
            parts = line.split()
            for part in parts:
                if "trycloudflare.com" in part and part.startswith("http"):
                    url = part.strip()
                    print("\n" + "="*60)
                    print(f"ACTIVE TUNNEL: {url}")
                    print("="*60 + "\n")
                    tunnel_established = True
                    break

threading.Thread(target=start_tunnel, daemon=True, args=(8188,)).start()

# 2. Launch Application
if os.path.exists(os.path.join(WORKSPACE, "main.py")):
    os.chdir(WORKSPACE)
    print(f"[SYSTEM] Booting ComfyUI [Profile: {MEMORY_PROFILE}]")
    print("[SYSTEM] Streaming Logs...\n" + "-"*60)

    cmd = ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"] + ARGS
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    for line in iter(process.stdout.readline, ''):
        sys.stdout.write(line)
        sys.stdout.flush()
else:
    print("[FATAL ERROR] ComfyUI directory missing. Please re-run the initialization cell.")

In [ ]:
#@title 5. Build Tarball Cache & Backup
#@markdown Compresses your local UI nodes into a single, high-speed archive and saves it to Drive. **Run this before disconnecting** so your next boot is instant.

import os
print("Compressing UI workspace into high-speed cache...")
WORKSPACE = "/content/ComfyUI"
CACHE_TAR = "/content/drive/MyDrive/ComfyUI/comfy_ui_cache.tar"

if os.path.exists(WORKSPACE):
    os.chdir(WORKSPACE)
    !tar -cf "{CACHE_TAR}" custom_nodes user
    print(f"[OK] Cache built and saved to: {CACHE_TAR}")
    print("[OK] You may now safely disconnect your session.")
else:
    print("[WARN] ComfyUI workspace not found. Nothing to backup.")